In this notebook, we collect the url of teams and players.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
from json import load as load_json

In [12]:
main_url = "https://www.basketball-reference.com/"
with open("config.json", 'r') as f:
        headers = load_json(f)
headers

{'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'}

In [4]:
def find_main_player_url(url: str, headers: dict):
        main_url_req = requests.get(
                url = url,
                headers = headers
        )
        soup = BeautifulSoup(main_url_req.text, "html.parser")

        a_tags = soup.select("#players > h2 > a")
        result = main_url + a_tags[0].get('href')[1:]
        soup.clear()
        return result

In [5]:
main_player_url = find_main_player_url(main_url, headers)

Players are seperated by their first letters. So we find a url for each letter and then use them to find every player's url.

In [6]:
def find_player_letter_urls(main_url: str, headers: dict, req_url: str, selector: str):
        NBA_player_url_req =  requests.get(
                url = req_url,
                headers = headers
        )

        soup = BeautifulSoup(NBA_player_url_req.text, "html.parser")
        li_tags = soup.select(selector = selector)
        letter_url = []
        for tag in li_tags:
                try:
                        letter_url.append(main_url + tag.a.get('href')[1:]) 
                except:
                        pass
        soup.clear()
        return letter_url


In [7]:
letter_url = find_player_letter_urls(main_url, headers, main_player_url, "#content > ul > li")
letter_url[:5]

['https://www.basketball-reference.com/players/a/',
 'https://www.basketball-reference.com/players/b/',
 'https://www.basketball-reference.com/players/c/',
 'https://www.basketball-reference.com/players/d/',
 'https://www.basketball-reference.com/players/e/']

Now we find urls for each player.

In [8]:
def find_player_info_urls(url: str, headers: dict):
        req =  requests.get(
                url = url,
                headers = headers
        )

        soup = BeautifulSoup(req.text, "html.parser")
        lst = soup.select(selector = "#players > tbody:nth-child(4) > tr:not(.thead)")
        players_lst = []
        for elem in lst:
                players_lst.append(main_url + elem.th.a.get("href")[1:])
        return players_lst

In [9]:
players_lst = []
for url in tqdm(letter_url):
        players_lst.extend(find_player_info_urls(url, headers))
        time.sleep(2)

print(f"{len(players_lst) = }")

100%|██████████| 25/25 [01:10<00:00,  2.82s/it]

len(players_lst) = 5416


At last, we save the result.

In [ ]:
with open("../../data/URLs/players_url.txt", "w") as f:
        f.write("\n".join(players_lst))

---
Now we find teams url.
Contrary to the structure of player urls, team urls are easier to find.

In [12]:
def find_team_info_urls(url: str, headers: dict):
        req =  requests.get(
                url = url,
                headers = headers
        )

        soup = BeautifulSoup(req.text, "html.parser")
        lst = soup.select(selector = "#players > tbody:nth-child(4) > tr:not(.thead)")
        tag_lst = soup.find_all(class_="full_table")
        team_lst = []
        for elem in tag_lst:
                team_lst.append(main_url + elem.th.a.get("href")[1:])
        return team_lst

In [13]:
team_url = "https://www.basketball-reference.com/teams/"
team_lst = find_team_info_urls(team_url, headers)


And now we save them.

In [ ]:
with open("../../data/URLs/teams_url.txt", "w") as f:
        f.write("\n".join(team_lst))